In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.pipeline import Pipeline

In [3]:
df = pd.read_csv('/kaggle/input/datasets/taweilo/loan-approval-classification-data/loan_data.csv')

In [4]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [6]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
num_cols = X_train.select_dtypes(['int','float']).columns
str_cols = X_train.select_dtypes('object').columns
num_cols_binary = [i for i in num_cols if X_train[i].max() <= 1 and X_train[i].min() >=0]
num_cols_numeric = [i for i in num_cols if i not in num_cols_binary]

In [9]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'), str_cols),
    ('num_scale', StandardScaler(), num_cols_numeric),
    ('num_unscale', 'passthrough', num_cols_binary)
])

In [10]:
model = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=42), random_state=42)

In [11]:
params = {
    'model__learning_rate' : [0.1, 0.5, 0.8],
    'model__n_estimators' : [100, 150, 200],
    'model__estimator__max_depth': [2, 4, 6]
}

In [12]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [13]:
final_model = GridSearchCV(pipe, params, cv=3, verbose=2, n_jobs=-1, scoring='roc_auc')

In [14]:
final_model.fit(X_train, y_train)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('categorical',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')),
                                                                        ('num_scale',
                                                                         StandardScaler(),
                                                                         ['person_age',
                                                                          'person_income',...
                                                                          'cb_person_cred_hist_length',
                                                                          'credit_score']),
                                                                        ('num_unscale',
                                                                         'passthrough',
                                                                         ['loan_percent_income'])])),
                                       ('model',
                                        AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=42),
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__estimator__max_depth': [2, 4, 6],
                         'model__learning_rate': [0.1, 0.5, 0.8],
                         'model__n_estimators': [100, 150, 200]},
             scoring='roc_auc', verbose=2)

In [34]:
y_proba = final_model.predict_proba(X_test)[:,1]

In [35]:
roc_auc_score(y_test, y_proba)

np.float64(0.9760312884789215)

In [36]:
final_model.best_score_

np.float64(0.9757364077528424)

In [37]:
y_pred = final_model.predict(X_test)

In [38]:
thresholds = np.arange(0.1,0.9,0.01)
f1 = [f1_score(y_test, y_proba > i) for i in thresholds]
best_threshold = thresholds[np.argmax(f1)]

In [42]:
y_pred = (y_proba >= best_threshold).astype(int)

In [43]:
confusion_matrix(y_test, y_pred)

array([[6676,  314],
       [ 313, 1697]])

In [44]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      6990
           1       0.84      0.84      0.84      2010

    accuracy                           0.93      9000
   macro avg       0.90      0.90      0.90      9000
weighted avg       0.93      0.93      0.93      9000



In [45]:
import joblib

In [46]:
joblib.dump(final_model.best_estimator_, 'best_model.joblib')

['best_model.joblib']